# 02 — Inside the embedding

What the Schmidt decomposition actually does, why N₂ correctly gets *no*
bath, and how to read the diagnostics.

Uses the validated reference pickles in `tests/regression/golden/`, so it
runs in seconds and needs no PySCF.

**One thing the pickles do not tell you:** ScH's was produced with a *forced*
active space (`force_active_space=[9, 10, 11, 12, 13, 14]`, giving `CAS(4e,6o)`),
because ASF under-selects for 3d elements. LiH and N₂ used automatic selection.

So when you compare `n_imp` across the three systems below, ScH's 6 impurity
orbitals were chosen by hand and the other two were not. See notebook 04
§"Forcing the active space" for how and why.

### Where these pickles come from

`tests/regression/golden/<system>/step2_hamiltonian.pkl` was produced on an
A100 / AMD EPYC 7402 run of `scripts/test_8`, July 2026. They are committed
deliberately: they let this notebook, the regression suite, and the
determinant-selection tools run with no PySCF and no GPU.

To regenerate one:

```bash
quenais-run --molecule LiH --basis sto-3g --steps 0 1 2 --project-dir ./lih_run --force
cp lih_run/results/step2_hamiltonian.pkl tests/regression/golden/LiH/
```

Do not overwrite a golden pickle without diffing it first —
`tools/compare_pickles.py` (last section of this notebook) exists for exactly
that, and a silent overwrite destroys the baseline that makes every regression
test meaningful.

### What is in a step-2 pickle

| key | meaning |
|---|---|
| `h1e`, `h2e` | embedded one- and two-electron integrals |
| `ecore` | frozen-core energy, fixed by requiring the embedding to reproduce the full-molecule mean-field energy |
| `n_alpha`, `n_beta` | embedded electron count **from the reference density**, not the active-space count |
| `n_imp`, `n_bath`, `n_emb` | impurity / bath / total embedded orbital counts. Qubits = 2 × `n_emb` |
| `sv_all` | the full Schmidt singular-value spectrum — the bath decision, before thresholding |
| `ref_occ_alpha`, `ref_occ_beta` | reference-density occupations, pre-rounding |
| `mu` | chemical potential, chosen so embedded particle number matches the reference |
| `embedded_scf_check` | the independent SCF-vs-UHF check |

`load_from_dmet_pickle()` (in `quenais.quantum.gqe_adapter`) requires
`h1e`, `h2e`, `ecore`, `n_alpha`, `n_beta` and raises a `KeyError` naming the
missing ones if the pickle predates them.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

GOLDEN = Path("../tests/regression/golden")

def load(system, stage):
    with open(GOLDEN / system / f"{stage}.pkl", "rb") as fh:
        return pickle.load(fh)

systems = ["LiH", "N2", "ScH"]
step2 = {s: load(s, "step2_hamiltonian") for s in systems}

## The Schmidt spectrum decides the bath

DMET splits the molecule into an impurity (the active space) and its
environment. The singular values of the impurity–environment block of the
reference density measure how entangled the two are. Large values mean
environment orbitals worth pulling into the calculation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, s in zip(axes, systems):
    sv = np.asarray(step2[s]["sv_all"])
    n_bath = step2[s]["n_bath"]
    colours = ["#C44E52" if i < n_bath else "#4C72B0" for i in range(len(sv))]
    ax.bar(range(len(sv)), sv, color=colours)
    ax.set_title(f"{s}: n_bath = {n_bath}")
    ax.set_xlabel("singular value index")
    print(f"{s:4}  max|sv| = {np.max(np.abs(sv)):.3e}   n_bath = {n_bath}")

axes[0].set_ylabel("Schmidt singular value")
fig.tight_layout()

## N₂ has no bath, and that is correct

Every singular value is around 1e-15 — numerically zero. The active
orbitals are already close to eigenvectors of the reference density, so
there is no impurity–environment entanglement left to extract.

The tolerance is `1e-8`. Nothing clears it, so the bath is empty and the
embedding is the active space alone.

Taking the largest values anyway — "there must be *some* bath" — builds
physics out of rounding noise. On N₂ that produced a badly non-orthonormal
embedding basis and roughly **20 Ha** of error.

**Provenance of that 20 Ha.** It is measured, not a figure of speech: it is the
error the fabricated-bath path produced on N₂/STO-3G, and it appears in
`docs/reproducibility.md` §2 and in the systematic-errors table in the thesis.
The important part is the second half of the sentence — *no convergence failure
of any kind*. The run completed, the SCF converged, and the number was wrong by
more than the total correlation energy.

The guard now is the tolerance itself (`bath_tolerance = 1e-8`), and the N₂ row
in the regression suite (`max_abs_sv_all_below: 1e-8`) is the tripwire that
keeps it guarded.

In [ ]:
from quenais.embedding.dmet_lib import adaptive_bath

sv_n2 = np.asarray(step2["N2"]["sv_all"])
print("N2 singular values:", sv_n2)
print()

n_bath, gap, cov = adaptive_bath(sv_n2, n_imp=4, max_embed=18, bath_tol=1e-8)
print(f"adaptive_bath -> n_bath={n_bath}, gap={gap}, coverage={cov}")

# What the pre-fix fallback would have done:
print(f"the old fallback would have taken the top {min(4, len(sv_n2))} anyway")

## The electron count comes from the density, not the active space

LiH's active space holds 2 electrons. Its *embedding* space — impurity
plus two bath orbitals — holds 4.

Deriving the count from the active space gives (1α, 1β). The reference
density says (2, 2). The wrong count roughly doubles the energy, and
nothing crashes.

In [ ]:
for s in systems:
    d2 = step2[s]
    a = float(np.sum(d2["ref_occ_alpha"]))
    b = float(np.sum(d2["ref_occ_beta"]))
    print(f"{s:4}  n_bath={d2['n_bath']}  "
          f"ref_occ sums = ({a:.6f}, {b:.6f})  "
          f"-> ({d2['n_alpha']}, {d2['n_beta']})")

## The independent check

`embedded_scf_check` compares a real SCF on the embedding Hamiltonian
against the full-molecule UHF energy. If the embedding Hamiltonian is
wrong, this fails — regardless of μ or the reference-density method.

In [ ]:
for s in systems:
    check = step2[s].get("embedded_scf_check")
    if check:
        print(f"{s:4}  delta = {check['delta']:+.3e} Ha   "
              f"within tolerance: {check['within_tol']}")
    else:
        print(f"{s:4}  (pickle predates the check)")

## Comparing your own run against these

`tools/compare_pickles.py` diffs any stage output against a golden one,
key by key, with per-quantity tolerances. It is what catches the failure
mode this project kept hitting: right shape, plausible magnitude, wrong
value.

```bash
python tools/compare_pickles.py \
    tests/regression/golden/LiH/step2_hamiltonian.pkl \
    ./lih_run/results/step2_hamiltonian.pkl -v
```

In [ ]:
import sys
sys.path.insert(0, "../tools")
from compare_pickles import compare

# Same file against itself: everything passes, nothing skipped.
report = compare(step2["LiH"], step2["LiH"])
print(report.summary())

# Now inject the electron-count bug and see it caught.
import copy
broken = copy.deepcopy(step2["LiH"])
broken["n_alpha"] = 1
print()
print(compare(step2["LiH"], broken).render())

---

## The bound that makes the method work

`n_bath ≤ n_imp`, always. This follows from the Schmidt decomposition: the
number of non-vanishing singular values is bounded by the *smaller* subsystem,
which is the impurity.

The practical consequence is the entire argument for embedding: **qubit count
is a property of the fragment, not of the molecule.** ScH needs 22 qubits
because its impurity is 6 orbitals — not because ScH has 22 electrons. Put the
same 6-orbital impurity in a molecule ten times larger and it still needs 22
qubits.

Check it against the numbers this notebook loaded:

| system | `n_imp` | `n_bath` | `n_emb` | qubits |
|---|---|---|---|---|
| LiH CAS(2e,2o) | 2 | 2 | 4 | 8 |
| ScH CAS(4e,6o) | 6 | 5 | 11 | 22 |
| N₂ CAS(4e,4o) | 4 | 0 | 4 | 8 |

## What "exact" does and does not mean here

The embedding is exact **with respect to the reference state used to build
it** — not with respect to the FCI ground state of the molecule. A poor
reference produces a bath that faithfully represents a poor state.

This is where the residual ScH embedding error comes from, and it is worth
being precise about it before someone else is: `reference_density_method` is
`casci` throughout, so the bath is exact for the CASCI reference, and the
CASCI reference is itself limited by the active space.

## Next

`05_determinant_selection` picks up from here: given a *correct* embedding
Hamiltonian, how much of the remaining error belongs to the sampler, and can
this system even tell selection methods apart?